# End-to-End Machine Learning Pipeline: E-Commerce Customer Risk Management

## Project Overview
The objective of this project is to develop an automated Machine Learning pipeline using **AWS SageMaker** to predict "customer risk" (the likelihood of severe negative order outcomes) for the Olist e-commerce platform. This is a batch-processing MLOps architecture designed to flag problematic transactions automatically, allowing for proactive intervention.

---

## Step 1: Data Sources
* **Source Dataset:** Olist Brazilian E-commerce Public Dataset (accessed via Kaggle).
* **Volume:** ~100,000 anonymized orders (2016-2018).
* **Ingestion:** Data is staged locally in the `datasets/` folder and uploaded to Amazon S3 (`s3://{sagemaker_default_bucket}/project/data/raw/`).
* **Structure:** Multiple relational tables including orders, customers, reviews, and payments.

In [1]:
!pip install "sagemaker<3.0.0" -q

In [45]:
import sagemaker
import boto3
import os

# Initialize SageMaker session, role, and bucket
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

# FIX: Remove the () from boto_region_name
region = session.boto_region_name

print(f"Execution Role: {role}")
print(f"Default Bucket: s3://{bucket}")
print(f"Region: {region}")

In [46]:
from sagemaker.workflow.parameters import ParameterString, ParameterFloat

raw_data_uri = ParameterString(name="RawDataUri", default_value=f"s3://{bucket}/project/data/raw/")
processed_data_uri = ParameterString(name="ProcessedDataUri", default_value=f"s3://{bucket}/project/data/processed/")
inference_output_uri = ParameterString(name="InferenceOutputUri", default_value=f"s3://{bucket}/project/inference/")
roc_auc_threshold = ParameterFloat(name="RocAucThreshold", default_value=0.75)

In [47]:
local_data_path = "datasets" 
raw_data_s3_uri = f"s3://{bucket}/project/data/raw"

print(f"Uploading files from local '{local_data_path}' to {raw_data_s3_uri}...")

uploaded_data_uri = sagemaker.s3.S3Uploader.upload(
    local_path=local_data_path,
    desired_s3_uri=raw_data_s3_uri,
    sagemaker_session=session
)

print(f"Data successfully uploaded to: {uploaded_data_uri}")

In [48]:
!aws s3 ls s3://sagemaker-us-east-1-781024672893/project/data/raw/

## Step 2 & 3: Data and Feature Engineering (Combined Processing Step)

In a SageMaker Pipeline, Data Engineering and Feature Engineering are executed together within a single `ProcessingStep`. Combining these phases is an MLOps best practice; it avoids the significant time and cost overhead of spinning up an expensive compute instance for data joining, shutting it down, and spinning up a new one just to calculate features. 

This pipeline step utilizes an `SKLearnProcessor` to execute our `preprocessing.py` script, which performs the following end-to-end data preparations:

* **Data Integration:** Loads raw CSV datasets mapped directly from S3 and merges the relational Olist tables (`orders`, `customers`, `payments`, `reviews`, and `items`) using their respective keys.
* **Target Label Generation:** Synthesizes the binary `is_high_risk` target label based on business logic (Review Score <= 2 AND logistical failures).
* **Feature Extraction:** * Calculates `delivery_delay_days` from timestamps.
    * Computes `freight_ratio` (freight cost divided by total payment value).
* **Data Transformation:** Handles missing null values and applies `LabelEncoder` to categorical variables (`payment_type`, `customer_state`) to ensure mathematical compatibility with the XGBoost algorithm.
* **Feature Store Ingestion:** Generates an `EventTime` stamp and ingests the fully engineered, identifiable records into the centralized SageMaker Feature Store (`olist-customer-risk-features`) for cross-team reusability.
* **Data Splitting & Formatting:** Drops unique identifiers (e.g., `order_id`), splits the dataset into an 80/20 Train/Test configuration, and saves the final output as headerless CSVs back to Amazon S3, exactly as required by the SageMaker XGBoost container.

In [49]:
%%writefile preprocessing.py
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def main():
    base_dir = '/opt/ml/processing'
    input_dir = os.path.join(base_dir, 'input')
    
    # 1. Load Data
    orders = pd.read_csv(os.path.join(input_dir, 'olist_orders_dataset.csv'))
    payments = pd.read_csv(os.path.join(input_dir, 'olist_order_payments_dataset.csv'))
    reviews = pd.read_csv(os.path.join(input_dir, 'olist_order_reviews_dataset.csv'))
    customers = pd.read_csv(os.path.join(input_dir, 'olist_customers_dataset.csv'))
    items = pd.read_csv(os.path.join(input_dir, 'olist_order_items_dataset.csv'))
    
    # 2. Merge
    df = orders.merge(customers, on='customer_id', how='left') \
               .merge(payments, on='order_id', how='left') \
               .merge(reviews, on='order_id', how='left')
    
    freight_df = items.groupby('order_id')['freight_value'].sum().reset_index()
    df = df.merge(freight_df, on='order_id', how='left').fillna(0)
    
    # 3. Feature Engineering
    datetime_cols = ['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date']
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        
    df['logistical_failure'] = ((df['order_status'] != 'delivered') | 
                                (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']))
    df['is_high_risk'] = np.where((df['review_score'] <= 2) & df['logistical_failure'], 1, 0)
    
    df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days.fillna(0)
    df['freight_ratio'] = np.where(df['payment_value'] > 0, df['freight_value'] / df['payment_value'], 0)
    
    # 4. Encoding
    le = LabelEncoder()
    df['payment_type'] = le.fit_transform(df['payment_type'].astype(str))
    df['customer_state'] = le.fit_transform(df['customer_state'].astype(str))
    
    # 5. Prepare for XGBoost
    cols = ['order_id', 'is_high_risk', 'payment_type', 'payment_installments', 'payment_value', 
            'freight_ratio', 'customer_state', 'delivery_delay_days']
    df_model = df[cols].copy().dropna()
    
    import time
    df_model['EventTime'] = pd.Series([time.time()] * len(df_model), dtype="float64")
    
    fs_dir = os.path.join(base_dir, 'fs_data')
    os.makedirs(fs_dir, exist_ok=True)
    df_model.to_csv(os.path.join(fs_dir, 'final_features.csv'), index=False)
    # -----------------------------------------------

    # Now strip the headers/IDs for XGBoost
    df_xgboost = df_model.drop(columns=['order_id', 'EventTime'])
    batch_data = df_xgboost.drop(columns=['is_high_risk']).tail(100)
    
    batch_dir = os.path.join(base_dir, 'batch')
    os.makedirs(batch_dir, exist_ok=True)
    
    # Save without headers/index as required by the XGBoost model
    batch_data.to_csv(os.path.join(batch_dir, 'batch.csv'), header=False, index=False)
    # ------------------------------------------

    train, test = train_test_split(df_xgboost, test_size=0.2, random_state=42, stratify=df_xgboost['is_high_risk'])
   
    # 6. Save
    os.makedirs(os.path.join(base_dir, 'train'), exist_ok=True)
    os.makedirs(os.path.join(base_dir, 'test'), exist_ok=True)
    train.to_csv(os.path.join(base_dir, 'train', 'train.csv'), header=False, index=False)
    test.to_csv(os.path.join(base_dir, 'test', 'test.csv'), header=False, index=False)

if __name__ == "__main__":
    main()

In [50]:
from sagemaker.workflow.parameters import ParameterString
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.functions import Join

# 1. Define the compute instance type parameter
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", 
    default_value="ml.m5.large"
)

# 2. Initialize the SKLearn Environment
# Use the latest supported Scikit-Learn container
sklearn_processor = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    # This helps ensure the container has the latest compatible libraries
    base_job_name="olist-processing",
    env={'PYTHONPATH': '/opt/ml/processing/input/code'}
)

# 3. Define the Processing Step
step_process = ProcessingStep(
    name="OlistDataAndFeatureEngineering",
    processor=sklearn_processor,
    inputs=[
        # Map the raw data in S3 to the container's input directory
        ProcessingInput(
            source=raw_data_uri, 
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        # Output the Train dataset
        ProcessingOutput(
            output_name='train', 
            source='/opt/ml/processing/train', 
            destination=Join(on="", values=[processed_data_uri, "train/"]) 
        ),
        # Output the Test dataset
        ProcessingOutput(
            output_name='test', 
            source='/opt/ml/processing/test', 
            destination=Join(on="", values=[processed_data_uri, "test/"])
        ),
        ProcessingOutput(
            output_name='feature_store',
            source='/opt/ml/processing/fs_data',
            destination=Join(on="", values=[processed_data_uri, "feature_store/"])
        ),
        ProcessingOutput(
            output_name='batch',
            source='/opt/ml/processing/batch',
            destination=Join(on="", values=[processed_data_uri, "batch/"])
        )
    ],
    # This executes the file we created using %%writefile preprocessing.py
    code='preprocessing.py' 
)

In [51]:
%%writefile ingest_features.py
import subprocess
import sys

# Force the container to install the SageMaker SDK before doing anything else
print("Installing sagemaker SDK...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "sagemaker<3.0.0", "-q"])
print("Installation complete.")

import os
import pandas as pd
import boto3
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup

def main():
    print("Starting Feature Store Ingestion Job...")
    
    # The pipeline will map the S3 output from step_process to this folder
    input_dir = '/opt/ml/processing/input'
    
    # Load the intact CSV that we saved in the previous step
    df = pd.read_csv(os.path.join(input_dir, 'final_features.csv'))
    
    # Initialize AWS sessions
    region = boto3.Session().region_name
    boto_session = boto3.Session(region_name=region)
    sagemaker_client = boto_session.client('sagemaker', region_name=region)
    featurestore_runtime = boto_session.client('sagemaker-featurestore-runtime', region_name=region)
    
    feature_store_session = sagemaker.Session(
        boto_session=boto_session, 
        sagemaker_client=sagemaker_client, 
        sagemaker_featurestore_runtime_client=featurestore_runtime
    )
    
    # Connect and Ingest
    feature_group_name = "olist-customer-risk-features"
    feature_group = FeatureGroup(name=feature_group_name, sagemaker_session=feature_store_session)
    
    # <--- THE FIX --->
    # Force the local object to download the schema from the AWS backend
    print("Loading feature group metadata from AWS...")
    feature_group.load_feature_definitions() 
    # <---------------->
    
    print(f"Ingesting {len(df)} records into {feature_group_name}...")
    feature_group.ingest(data_frame=df, max_workers=3, wait=True)
    print("Ingestion complete!")

if __name__ == "__main__":
    main()

In [52]:
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput
from sagemaker.workflow.steps import ProcessingStep

# 1. Retrieve the built-in XGBoost image URI 
xgboost_image = sagemaker.image_uris.retrieve("xgboost", region, "1.7-1")

# 2. Define a Processor with the Region explicitly passed in!
fs_processor = ScriptProcessor(
    image_uri=xgboost_image, 
    command=['python3'],
    role=role,
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="olist-fs-ingest",
    env={
        'PIP_INSTALL': 'sagemaker',
        'AWS_DEFAULT_REGION': region 
    } 
)

# 3. Define the Ingestion Step
step_fs_ingest = ProcessingStep(
    name="OlistFeatureStoreIngestion",
    processor=fs_processor,
    inputs=[
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["feature_store"].S3Output.S3Uri,
            destination='/opt/ml/processing/input'
        )
    ],
    code="ingest_features.py"
)

## Step 4: Model Training & Evaluation
* **Algorithm:** SageMaker's built-in `XGBoost Classifier`, handling non-linear relationships and configured to address class imbalance (`scale_pos_weight`).
* **Compute:** Dedicated `ml.m5.xlarge` instance for training workloads.
* **Experiment Tracking:** Hyperparameters and metrics are logged continuously using MLflow.
* **Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, and ROC-AUC. Given the business cost of false positives vs. false negatives, balancing Precision and Recall is the primary analytical objective.

In [53]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep
import sagemaker

# Retrieve the built-in XGBoost image URI
xgboost_image = sagemaker.image_uris.retrieve("xgboost", region, "1.7-1")

# Configure the XGBoost Estimator
xgb_estimator = Estimator(
    image_uri=xgboost_image,
    instance_type="ml.m5.large",
    instance_count=1,
    role=role,
    output_path=f"s3://{bucket}/project/models/",
    hyperparameters={
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "max_depth": "6",
        "eta": "0.05",
        "scale_pos_weight": "5.0", # Addresses the High/Low risk imbalance
        "num_round": "100"
    }
)

# Define the Training Step
step_train = TrainingStep(
    name="OlistXGBoostTraining",
    estimator=xgb_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

In [54]:
%%writefile evaluate.py
#!/usr/bin/env python3
import os
import json
import tarfile
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def main():
    base_dir = '/opt/ml/processing'
    model_path = os.path.join(base_dir, 'model', 'model.tar.gz')
    test_path = os.path.join(base_dir, 'test', 'test.csv')
    output_dir = os.path.join(base_dir, 'evaluation')
    
    print("Extracting model...")
    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")
    
    print("Loading model...")
    # SageMaker built-in XGBoost saves the model as 'xgboost-model'
    model = xgb.Booster()
    model.load_model("xgboost-model")
    
    print("Loading test data...")
    # Read test data (no headers, target is the first column)
    test_df = pd.read_csv(test_path, header=None)
    y_test = test_df.iloc[:, 0].values
    X_test = test_df.iloc[:, 1:].values
    
    dtest = xgb.DMatrix(X_test)
    
    print("Generating predictions...")
    predictions_proba = model.predict(dtest)
    # Default threshold of 0.5 for binary classification
    predictions = np.where(predictions_proba > 0.5, 1, 0)
    
    print("Calculating metrics...")
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, predictions_proba)
    
    report_dict = {
        "classification_metrics": {
            "accuracy": {"value": accuracy},
            "precision": {"value": precision},
            "recall": {"value": recall},
            "f1_score": {"value": f1},
            "roc_auc": {"value": roc_auc}
        }
    }
    
    print("Saving evaluation report...")
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "evaluation.json"), "w") as f:
        f.write(json.dumps(report_dict))
        
    print(f"Evaluation complete. ROC-AUC: {roc_auc:.4f}, F1-Score: {f1:.4f}")

if __name__ == "__main__":
    main()

In [55]:
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

# 1. Define the PropertyFile (This fixes your NameError!)
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

# 2. Initialize the ScriptProcessor (with our python3 fix)
xgb_script_processor = ScriptProcessor(
    image_uri=xgboost_image,              
    command=['python3'],                  
    role=role,
    instance_type=processing_instance_type, 
    instance_count=1,
    base_job_name="olist-evaluation"
)

# 3. Define the Evaluation Step
step_eval = ProcessingStep(
    name="OlistModelEvaluation",
    processor=xgb_script_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation", 
            source="/opt/ml/processing/evaluation"
        )
    ],
    code="evaluate.py",
    property_files=[evaluation_report] 
)

## Step 5: Model Deployment
* **Model Registry:** The trained model is registered to the SageMaker Model Registry (`Olist-Risk-Model-Group`) with an initial status of `PendingManualApproval`.
* **Conditional Registration:** The pipeline uses a `ConditionStep` to ensure registration only occurs if the model meets a strict baseline threshold (e.g., ROC-AUC >= 0.75).
* **Batch Inference:** Approved models are deployed via a SageMaker Batch Transform job to score daily unlabelled transactions. Predictions are saved directly to S3 (`s3://{sagemaker_default_bucket}/project/inference/`) for consumption by BI teams.

In [56]:
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.functions import Join
from sagemaker.workflow.pipeline_context import PipelineSession # <-- New Import

# Initialize a PipelineSession
pipeline_session = PipelineSession()

# 1. Map the evaluation metrics to the Model Registry
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/", 
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri, 
                "evaluation.json"
            ]
        ),
        content_type="application/json"
    )
)

# 2. Create a formal Model object from the training output
model = Model(
    image_uri=xgboost_image,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session, # <-- FIX: Use PipelineSession here
    role=role
)

# 3. Define the Model Registration Step using the Model object
# 3. Define the Model Registration Step using the Model object
step_register = ModelStep(
    name="RegisterOlistRiskModel", # This is the unique name for the pipeline step
    step_args=model.register(
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name="Olist-Risk-Model-Group",
        approval_status="PendingManualApproval",
        model_metrics=model_metrics
    )
)

In [57]:
from sagemaker.transformer import Transformer
from sagemaker.workflow.steps import TransformStep
from sagemaker.inputs import TransformInput
from sagemaker.workflow.model_step import ModelStep

# 1. NEW: Create the Model resource in SageMaker
step_create_model = ModelStep(
    name="CreateOlistModel",
    step_args=model.create(instance_type="ml.m5.large")
)

# 2. Initialize the Transformer
# Point it to the exact ModelName generated by the step above!
transformer = Transformer(
    model_name=step_create_model.properties.ModelName, # <--- The Magic Fix
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=inference_output_uri,
    assemble_with="Line",
    accept="text/csv"
)

# 3. Define the Transform Step
step_transform = TransformStep(
    name="OlistBatchRiskScoring",
    transformer=transformer,
    inputs=TransformInput(
        data=Join(on="", values=[processed_data_uri, "batch/batch.csv"]),
        content_type="text/csv",
        split_type="Line"
    )
)

In [58]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo

# 1. Define the Condition (ROC-AUC >= Threshold)
condition_roc = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name="OlistModelEvaluation",
        property_file=evaluation_report,
        json_path="classification_metrics.roc_auc.value"
    ),
    right=roc_auc_threshold
)

# 2. Define the Condition Step
# Add step_create_model to the if_steps array so it executes upon success
step_cond = ConditionStep(
    name="CheckRocAucThreshold",
    conditions=[condition_roc],
    if_steps=[step_register, step_create_model, step_transform], # <--- Added here!
    else_steps=[]
)

# 3. Construct the SageMaker Pipeline
pipeline = Pipeline(
    name="Olist-Automated-Risk-Pipeline",
    parameters=[
        raw_data_uri, 
        processed_data_uri, 
        inference_output_uri, 
        roc_auc_threshold,
        processing_instance_type
    ],
    # The DAG (Directed Acyclic Graph) is defined by passing the steps in order
    steps=[step_process, step_fs_ingest,step_train, step_eval, step_cond]
)

# 4. Upsert (Create or Update) and Execute
print("Upserting pipeline definition to Amazon SageMaker...")
pipeline.upsert(role_arn=role)

print("Starting pipeline execution...")
execution = pipeline.start()

# Print the execution ARN so you can track it in the AWS Console / SageMaker Studio UI
print(f"Execution ARN: {execution.arn}")

# Optional: execution.wait() blocks the notebook cell until the pipeline succeeds or fails

## Step 6: Model Monitoring
* **Model Quality Drift:** SageMaker Model Monitor compares scheduled batch predictions against actual outcomes once ground truth labels (e.g., actual chargebacks) mature.
* **Data Quality Drift:** Monitors statistical baselines of features to detect shifts in input distributions (e.g., sudden regional spikes or payment method changes).
* **Infrastructure Monitoring:** Amazon CloudWatch tracks CPU/Memory utilization and the execution duration of the batch transform jobs.

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

# 1. Initialize the Data Quality Monitor
data_quality_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
)

baseline_results_uri = f"s3://{bucket}/project/monitoring/data-quality-baseline"

print("Starting Data Quality Baseline Job...")
# 2. Run the baseline job against the training data generated in Step 2
data_quality_monitor.suggest_baseline(
    baseline_dataset=f"{processed_data_uri.default_value}train/train.csv",
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=baseline_results_uri,
    wait=True
)

print(f"Data Quality Baseline generated and saved to: {baseline_results_uri}")

In [ ]:
from sagemaker.model_monitor import CronExpressionGenerator

monitoring_reports_uri = f"s3://{bucket}/project/monitoring/data-quality-reports"

# Create a monitoring schedule to run daily
print("Creating daily Data Quality Monitoring Schedule...")
data_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name="Olist-Daily-Data-Monitor",
    batch_transform_input=f"{processed_data_uri.default_value}batch/batch.csv",
    output_s3_uri=monitoring_reports_uri,
    statistics=f"{baseline_results_uri}/statistics.json",
    constraints=f"{baseline_results_uri}/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics=True # Sends drift metrics directly to CloudWatch
)

print("Monitoring schedule activated. Drift metrics will be available in CloudWatch.")

In [ ]:
import boto3

cloudwatch = boto3.client('cloudwatch', region_name=region)

alarm_name = "Olist-Batch-Transform-High-CPU"

print("Configuring CloudWatch Infrastructure Alarm...")
cloudwatch.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription='Alarm when Batch Transform CPU exceeds 90% for 2 consecutive periods',
    ActionsEnabled=False, # Set to True and add an SNS Topic ARN to enable email/Slack alerts
    MetricName='CPUUtilization',
    Namespace='AWS/SageMaker',
    Statistic='Average',
    Dimensions=[
        {
            'Name': 'TransformJobName',
            # You would dynamically pass the name of the active transform job here
            'Value': 'OlistBatchRiskScoring' 
        },
    ],
    Period=300, # Evaluate every 5 minutes
    EvaluationPeriods=2,
    Threshold=90.0,
    ComparisonOperator='GreaterThanThreshold',
    TreatMissingData='missing'
)

print(f"CloudWatch Alarm '{alarm_name}' configured successfully.")

## Step 7: CI/CD Integration
In this project, Continuous Integration and Continuous Deployment (CI/CD) principles were implemented using AWS SageMaker Pipelines and native version control:

* **Source Control:** All project artifacts, including this Jupyter Notebook and the decoupled Python execution scripts (`preprocessing.py`, `evaluate.py`, and `ingest_features.py`), are version-controlled within our integrated Git repository. This ensures reproducibility and facilitates collaborative code review.
* **Pipeline Automation:** We transitioned from localized notebook execution to a fully automated MLOps architecture using the SageMaker SDK. By defining a Directed Acyclic Graph (DAG) named `Olist-Automated-Risk-Pipeline`, we automated the sequential triggering of data engineering, feature store ingestion, XGBoost model training, and performance evaluation steps without human intervention.
* **Approval Gates:** To simulate a production-grade governance structure, we implemented a "human-in-the-loop" approval gate. Within our `RegisterOlistRiskModel` step, the model is registered to the `Olist-Risk-Model-Group` with a strict `PendingManualApproval` status. It cannot be passed to the batch inference transform job until an administrator manually reviews the validation metrics and approves it in the SageMaker Model Registry UI.

In [ ]:
import boto3

sm_client = boto3.client('sagemaker')

# The exact job name from your notebook logs
job_name = 'baseline-suggestion-job-2026-06-19-13-20-23-640'

try:
    sm_client.stop_processing_job(ProcessingJobName=job_name)
    print(f"Successfully sent stop signal to: {job_name}")
except Exception as e:
    print(f"Job may have already stopped or failed. Details: {e}")